In [ ]:
import os
import sys
sys.path.append("../..")
import copy
import csv
import numpy as np
from scipy import linalg as LA
import pandas as pd
from google.colab import files

def read_country(filename):
    f = open(filename,'r', encoding='gbk')
    reader = csv.reader(f)
    raw_matrix = []
    for row in reader:
        temp = []
        for i in range(0,len(row)):
            try:
                temp.append(float(row[i]))
            except ValueError:
                temp.append(row[i])
        raw_matrix.append(temp)  
    f.close()
    return raw_matrix


def eci(raw_matrix,countryname):

    
    #compute mcp
    product_list = raw_matrix[0][1::]
    country_list = []
    for i in range(1,len(raw_matrix)):
        country_list.append(raw_matrix[i][0])
    
    country_sum = []
    for i in range(1,len(raw_matrix)):
        country_sum.append(np.sum(raw_matrix[i][1::]))
    
    product_sum = []
    for i in range(1,len(raw_matrix[0])):
        product = 0
        for j in range(1,len(raw_matrix)):
            product += raw_matrix[j][i]
        product_sum.append(product)
    total_sum = np.sum(product_sum)
    
    mcp = []
    
    for i in range (1,len(raw_matrix)):
        country_row = []
        for j in range(1,len(raw_matrix[i])):
            percentage = raw_matrix[i][j]/country_sum[i-1]
            total_percentage = product_sum[j-1]/total_sum
            if percentage > total_percentage:
                country_row.append(1)
            else:
                country_row.append(0)
        mcp.append(country_row)
    
    #compute mcc
    mcp2 = mcp
    country_sum = []
    for i in range(0,len(mcp)):
        country_sum.append(np.sum(mcp[i]))
    
    product_sum = []
    for i in range(0,len(mcp[0])):
        product = 0
        for j in range(0,len(mcp)):
            product += mcp[j][i]
        product_sum.append(product)
    total_sum = np.sum(product_sum)
    
    mcc = []
    for i in range(0,len(mcp)):
        country_row = []
        for j in range(0,len(mcp)):
            temp_sum = 0
            for k in range(0,len(mcp[0])):
                try:
                    temp_sum += (mcp[i][k] * mcp2[j][k]/(country_sum[i]*product_sum[k]))
                except:
                    pass
            country_row.append(temp_sum)
        mcc.append(country_row)
    
    #compute and write ECI
    mcc = np.array(mcc)
    
    e_vals, e_vecs = LA.eig(mcc)
    eigenvectors = e_vecs.tolist()
    eigen = []
    for eigenvector in eigenvectors:
        temp = []
        for col in eigenvector:
            temp.append(col.real)
        eigen.append(temp)
    
    vector = []
    for i in range(0,len(eigen)):
        vector.append(eigen[i][1])
    
    eci = []
    if vector[1] < 0:
        for item in vector:
            eci.append((item-np.mean(vector))/np.std(vector))
    else:
        for item in vector:
            eci.append((-item+np.mean(vector))/np.std(vector))
    country=list(zip(*raw_matrix))[0][1:]
    ind=country.index(countryname) 
    return eci[ind]
    


In [ ]:
def oppo(raw_matrix,countryname):
    #Read the raw matrix
    #compute mcp
    product_list = raw_matrix[0][1::]
    country_list = []
    for i in range(1,len(raw_matrix)):
        country_list.append(raw_matrix[i][0])
    
    country_sum = []
    for i in range(1,len(raw_matrix)):
        country_sum.append(np.sum(raw_matrix[i][1::]))
    
    product_sum = []
    for i in range(1,len(raw_matrix[0])):
        product = 0
        for j in range(1,len(raw_matrix)):
            product += raw_matrix[j][i]
        product_sum.append(product)
    total_sum = np.sum(product_sum)
    
    mcp = []
    
    for i in range (1,len(raw_matrix)):
        country_row = []
        for j in range(1,len(raw_matrix[i])):
            percentage = raw_matrix[i][j]/country_sum[i-1]
            total_percentage = product_sum[j-1]/total_sum
            if percentage > total_percentage:
                country_row.append(1)
            else:
                country_row.append(0)
        mcp.append(country_row)
    
    #compute mpp
    country_sum = []
    for i in range(0,len(mcp)):
        country_sum.append(np.sum(mcp[i]))
    
    product_sum = []
    for i in range(0,len(mcp[0])):
        product = 0
        for j in range(0,len(mcp)):
            product += mcp[j][i]
        product_sum.append(product)
    total_sum = np.sum(product_sum)
    
    mpp = []
    for i in range(0,len(mcp[0])):
        product_row = []
        for j in range(0,len(mcp[0])):
            temp_sum = 0
            for k in range(0,len(mcp)):
                try:
                    temp_sum += (mcp[k][i] * mcp[k][j]/(country_sum[k]*product_sum[i]))
                except:
                    pass
            product_row.append(temp_sum)
        mpp.append(product_row)
    
    #compute PCI
    mpp = np.array(mpp)
    
    e_vals, e_vecs = LA.eig(mpp)
    eigenvectors = e_vecs.tolist()
    eigen = []
    for eigenvector in eigenvectors:
        temp = []
        for col in eigenvector:
            temp.append(col.real)
        eigen.append(temp)
    
    vector = []
    for i in range(0,len(eigen)):
        vector.append(eigen[i][1])
    
    pci = []
    if vector[len(vector)-2] > 0:
        for item in vector:
            pci.append((item-np.mean(vector))/np.std(vector))
    else:
        for item in vector:
            pci.append((-item+np.mean(vector))/np.std(vector))
    
    #compute proximity
    proximity = []
    for i in range(0,len(mcp[0])):
        product_row = []
        for j in range(0,len(mcp[0])):
            temp_sum = 0
            for k in range(0,len(mcp)):
                try:
                    temp_sum += mcp[k][i] * mcp[k][j]
                except:
                    pass
            p0 = product_sum[i]
            p1 = product_sum[j]
            if p0 < p1:
                large = p1
            else:
                large = p0
            product_row.append(temp_sum/large)
        proximity.append(product_row)
    
    #compute actual distance
    distance = []
    for i in range(0,len(mcp)):
        product_row = []
        for j in range(0,len(mcp[0])):
            sum1 = 0
            for k in range(0,len(mcp[0])):
                try:
                    sum1 += (1-mcp[i][k])*proximity[j][k]
                except:
                    pass
            sum2 = 0
            for k in range(0,len(mcp[0])):
                try:
                    sum2 += proximity[j][k]
                except:
                    pass
            if mcp[i][j] == 1:
                product_row.append(0)
            else:
                product_row.append(sum1/sum2)
        distance.append(product_row)
    
    #compute opportunity gain
    opportunity_gain = []
    for i in range(0,len(mcp)):
        country_row = []
        for j in range(0,len(mcp[0])):
            gain = 0
            for k in range(0,len(mcp[0])):
                temp = 0
                for l in range(0,len(mcp[0])):
                    temp += proximity[l][k]
                gain += (proximity[j][k]/temp)*(1-mcp[i][k])*pci[k]
            gain = gain - (1-distance[i][j]) * pci[j]
            if mcp[i][j] == 1:
                country_row.append(0)
            else:
                country_row.append(gain)
        opportunity_gain.append(country_row)
    
    opportunity_gain = np.array(opportunity_gain)
    og = np.transpose(opportunity_gain)
    og=pd.DataFrame(og)
    og.index=raw_matrix[0][1:]
    og.columns=list(zip(*raw_matrix))[0][1:]
    og=pd.DataFrame(og[countryname].sort_values(ascending=False)) 
    og['new']=og.index
    boo=og[countryname]==0
    temp=og.copy()
    og['new'][boo]="zero:"+temp['new'][boo]
    og.index=np.arange(18)
    return og

In [ ]:

def sensitivity(raw,countryname,industryname,folder):
    percent1=[1,0.9,0.8,0.7,0.6,0.5]
    percent2=[1,1.5,2,2.5,3,3.5]
    country=list(zip(*raw))[0]
    ind=country.index(countryname) 
    industry=raw[0]
    indc=industry.index(industryname)
    df1=pd.DataFrame()
    df2=pd.DataFrame()
    df3=pd.DataFrame()
    for i in percent1:
        raw_matrix=copy.deepcopy(raw)
        raw_matrix[ind][indc]=raw_matrix[ind][indc]*i
        k=oppo(raw_matrix,countryname)
        m=k["new"]
        m=m.append(pd.Series(eci(raw_matrix,countryname),index=['ECI']))
        df1[str(i)]=m
    for i in percent2:
        raw_matrix=copy.deepcopy(raw)
        raw_matrix[ind][indc]=raw_matrix[ind][indc]*i
        k=oppo(raw_matrix,countryname)
        m=k["new"]
        m=m.append(pd.Series(eci(raw_matrix,countryname),index=['ECI']))
        df2[str(i)]=m
        
    
  
    with pd.ExcelWriter(folder+countryname+industryname+ ".xlsx") as writer:  # doctest: +SKIP
        df1.to_excel(writer, sheet_name='Sheet_name_1')
        df2.to_excel(writer, sheet_name='Sheet_name_2')
    files.download(folder+countryname+industryname+ ".xlsx")
    print("done")

In [ ]:
def estimate_coef(x, y): 
    # number of observations/points 
    n = np.size(x) 
  
    # mean of x and y vector 
    m_x, m_y = np.mean(x), np.mean(y) 
  
    # calculating cross-deviation and deviation about x 
    SS_xy = np.sum(y*x) - n*m_y*m_x 
    SS_xx = np.sum(x*x) - n*m_x*m_x 
  
    # calculating regression coefficients 
    b_1 = SS_xy / SS_xx 
    b_0 = m_y - b_1*m_x 
  
    return(b_1)

def coef(raw_matrix,countryname,industryname1,industryname2,folder):
    percent1=np.array([1,0.9,0.8,0.7,0.6,0.5,0.4,0.3])-1
    percent2=np.array([1,1.5,2,2.5,3,3.5,4,4.5])-1
    country=list(zip(*raw_matrix))[0]
    ind=country.index(countryname) 
    industry=raw_matrix[0]
    indc1=industry.index(industryname1)
    indc2=industry.index(industryname2)
    ECI11=[]
    ECI12=[]
    ECI21=[]
    ECI22=[]
    for i in percent1:
        raw=copy.deepcopy(raw_matrix)
        raw[ind][indc1]=raw[ind][indc1]*(i+1)
        ECI11=np.append(ECI11,eci(raw,countryname))
        
    for i in percent2:
        raw=copy.deepcopy(raw_matrix)
        raw[ind][indc1]=raw[ind][indc1]*(i+1)
        ECI12=np.append(ECI12,eci(raw,countryname))
        
    for i in percent1:
        raw=copy.deepcopy(raw_matrix)
        raw[ind][indc2]=raw[ind][indc2]*(i+1)
        ECI21=np.append(ECI21,eci(raw,countryname))
        
    for i in percent2:
        raw=copy.deepcopy(raw_matrix)
        raw[ind][indc2]=raw[ind][indc2]*(i+1)
        ECI22=np.append(ECI22,eci(raw,countryname))
    
    result=pd.DataFrame()
    result[industryname1]=pd.Series([estimate_coef(percent1,ECI11),estimate_coef(percent2,ECI12)],index=['Decremental','Incremental'])
    result[industryname2]=pd.Series([estimate_coef(percent1,ECI21),estimate_coef(percent2,ECI22)],index=['Decremental','Incremental'])
    with pd.ExcelWriter(folder+countryname+ "_coef.xlsx") as writer:  # doctest: +SKIP
        result.to_excel(writer, sheet_name='Sheet_name_1')
    print(countryname)

In [ ]:
def cont(countryname,raw_matrix,folder):
    df1=pd.DataFrame()
    country=list(zip(*raw_matrix))[0]
    ind=country.index(countryname) 
    industry=raw_matrix[0]
    for inds in  industry[1:]:
        raw=copy.deepcopy(raw_matrix)
        indc=industry.index(inds)
        raw[ind][indc]=raw[ind][indc]*0.3
        temp=oppo(raw,countryname)['new'].iloc[0:3]
        t=pd.Series(temp)
        df1=df1.append(t)
    df1.index=industry[1:]
    with pd.ExcelWriter(folder+countryname + "_contingency.xlsx") as writer:  # doctest: +SKIP
        df1.to_excel(writer, sheet_name='Sheet_name_1')
    print("done")

# Read in Data for 2014

In [ ]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)
root_path = 'drive/My Drive/ECI Research/Spring 2020/Data/'
raw_matrix=read_country(root_path+'2013-2018 cleaned/2013clean.csv')

Mounted at /content/drive/


# All Sensitivity-----2014

In [ ]:

sensitivity(raw_matrix,"Germany","High-complexity Products", root_path+'2013-2018 cleaned/')
sensitivity(raw_matrix,"Germany","Food.and.preparation",'2014 Excels/')
"""
sensitivity(raw_matrix,"France","High.tech",'2014 Excels/')
sensitivity(raw_matrix,"France","Food.and.preparation",'2014 Excels/')

sensitivity(raw_matrix,"USA","High.tech",'2014 Excels/')
sensitivity(raw_matrix,"USA","Food.and.preparation",'2014 Excels/')

sensitivity(raw_matrix,"Malaysia","High.tech",'2014 Excels/')
sensitivity(raw_matrix,"Malaysia","Food.and.preparation",'2014 Excels/')

sensitivity(raw_matrix,"China","High.tech",'2014 Excels/')
sensitivity(raw_matrix,"China","Food.and.preparation",'2014 Excels/')

sensitivity(raw_matrix,"Mexico","High.tech",'2014 Excels/')
sensitivity(raw_matrix,"Mexico","Food.and.preparation",'2014 Excels/')

sensitivity(raw_matrix,"Japan","High.tech",'2014 Excels/')
sensitivity(raw_matrix,"Japan","Food.and.preparation",'2014 Excels/')

sensitivity(raw_matrix,'Rep. of Korea',"High.tech",'2014 Excels/')
sensitivity(raw_matrix,'Rep. of Korea',"Food.and.preparation",'2014 Excels/')"""

In [ ]:
cont('France',raw_matrix,'2014 Excels/')
cont('China',raw_matrix,'2014 Excels/')
cont('Mexico',raw_matrix,'2014 Excels/')
cont('Japan',raw_matrix,'2014 Excels/')

cont('USA',raw_matrix,'2014 Excels/')
cont('Germany',raw_matrix,'2014 Excels/')
cont('Malaysia',raw_matrix,'2014 Excels/')
cont('Rep. of Korea',raw_matrix,'2014 Excels/')

# All coefficients---2014

In [ ]:
raw_matrix=read_country("2014matched.csv")
coef(raw_matrix,'China','High.tech','Food.and.preparation','2014 Excels/')
coef(raw_matrix,'USA','High.tech','Food.and.preparation','2014 Excels/')
coef(raw_matrix,'Japan','High.tech','Food.and.preparation','2014 Excels/')
coef(raw_matrix,'Mexico','High.tech','Food.and.preparation','2014 Excels/')

coef(raw_matrix,'Germany','High.tech','Food.and.preparation','2014 Excels/')
coef(raw_matrix,'France','High.tech','Food.and.preparation','2014 Excels/')
coef(raw_matrix,'Malaysia','High.tech','Food.and.preparation','2014 Excels/')
coef(raw_matrix,'Rep. of Korea','High.tech','Food.and.preparation','2014 Excels/')


China
USA
Japan
Mexico
Germany
France
Malaysia
Rep. of Korea


# Benchmark ---2014

In [ ]:
raw_matrix=read_country("2014matched.csv")
raw_matrix=np.array(raw_matrix)
country=list(zip(*raw_matrix))[0]
c_list=np.array(['Japan','Mexico','USA','China','Germany','France','Malaysia','Rep. of Korea'])
flag=np.isin(np.array(raw_matrix[:,0]),c_list)
l=np.mean(raw_matrix[flag,1:].astype(float),axis=0)   
raw_matrix[1,1:]=l
raw_matrix[1,0]="Benchmark"
raw_matrix=raw_matrix.tolist()
for n in np.arange(1,119,1):
    raw_matrix[n][1:]=[float(i) for i in raw_matrix[n][1:]]

sensitivity(raw_matrix,"Benchmark","High.tech",'2014 Excels/')
sensitivity(raw_matrix,"Benchmark","Food.and.preparation",'2014 Excels/')
cont('Benchmark',raw_matrix,'2014 Excels/')
coef(raw_matrix,'Benchmark','High.tech','Food.and.preparation','2014 Excels/')

C:\Users\Administrator\Anaconda3\lib\site-packages\ipykernel_launcher.py:154: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy


done
done
done
Benchmark


# Read in Data for 2009

In [ ]:
raw_matrix=read_country('2009matched.csv')

# All Sensitivity-----2009

In [ ]:

sensitivity(raw_matrix,"Germany","High.tech",'2009 Excels/')
sensitivity(raw_matrix,"Germany","Food.and.preparation",'2009 Excels/')

sensitivity(raw_matrix,"France","High.tech",'2009 Excels/')
sensitivity(raw_matrix,"France","Food.and.preparation",'2009 Excels/')

sensitivity(raw_matrix,"USA","High.tech",'2009 Excels/')
sensitivity(raw_matrix,"USA","Food.and.preparation",'2009 Excels/')

sensitivity(raw_matrix,"Malaysia","High.tech",'2009 Excels/')
sensitivity(raw_matrix,"Malaysia","Food.and.preparation",'2009 Excels/')

sensitivity(raw_matrix,"China","High.tech",'2009 Excels/')
sensitivity(raw_matrix,"China","Food.and.preparation",'2009 Excels/')

sensitivity(raw_matrix,"Mexico","High.tech",'2009 Excels/')
sensitivity(raw_matrix,"Mexico","Food.and.preparation",'2009 Excels/')

sensitivity(raw_matrix,"Japan","High.tech",'2009 Excels/')
sensitivity(raw_matrix,"Japan","Food.and.preparation",'2009 Excels/')

sensitivity(raw_matrix,'Rep. of Korea',"High.tech",'2009 Excels/')
sensitivity(raw_matrix,'Rep. of Korea',"Food.and.preparation",'2009 Excels/')

C:\Users\Administrator\Anaconda3\lib\site-packages\ipykernel_launcher.py:154: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy


done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done


In [ ]:
cont('France',raw_matrix,'2009 Excels/')
cont('China',raw_matrix,'2009 Excels/')
cont('Mexico',raw_matrix,'2009 Excels/')
cont('Japan',raw_matrix,'2009 Excels/')

cont('USA',raw_matrix,'2009 Excels/')
cont('Germany',raw_matrix,'2009 Excels/')
cont('Malaysia',raw_matrix,'2009 Excels/')
cont('Rep. of Korea',raw_matrix,'2009 Excels/')

C:\Users\Administrator\Anaconda3\lib\site-packages\ipykernel_launcher.py:154: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy


done
done
done
done
done
done
done
done


# All coefficients---2009

In [ ]:
raw_matrix=read_country("2009matched.csv")
coef(raw_matrix,'China','High.tech','Food.and.preparation','2009 Excels/')
coef(raw_matrix,'USA','High.tech','Food.and.preparation','2009 Excels/')
coef(raw_matrix,'Japan','High.tech','Food.and.preparation','2009 Excels/')
coef(raw_matrix,'Mexico','High.tech','Food.and.preparation','2009 Excels/')

coef(raw_matrix,'Germany','High.tech','Food.and.preparation','2009 Excels/')
coef(raw_matrix,'France','High.tech','Food.and.preparation','2009 Excels/')
coef(raw_matrix,'Malaysia','High.tech','Food.and.preparation','2009 Excels/')
coef(raw_matrix,'Rep. of Korea','High.tech','Food.and.preparation','2009 Excels/')


China
USA
Japan
Mexico
Germany
France
Malaysia
Rep. of Korea


# Benchmark ---2009

In [ ]:
raw_matrix=read_country("2009matched.csv")
raw_matrix=np.array(raw_matrix)
country=list(zip(*raw_matrix))[0]
c_list=np.array(['Japan','Mexico','USA','China','Germany','France','Malaysia','Rep. of Korea'])
flag=np.isin(np.array(raw_matrix[:,0]),c_list)
l=np.mean(raw_matrix[flag,1:].astype(float),axis=0)   
raw_matrix[1,1:]=l
raw_matrix[1,0]="Benchmark"
raw_matrix=raw_matrix.tolist()
for n in np.arange(1,119,1):
    raw_matrix[n][1:]=[float(i) for i in raw_matrix[n][1:]]

sensitivity(raw_matrix,"Benchmark","High.tech",'2009 Excels/')
sensitivity(raw_matrix,"Benchmark","Food.and.preparation",'2009 Excels/')
cont('Benchmark',raw_matrix,'2009 Excels/')
coef(raw_matrix,'Benchmark','High.tech','Food.and.preparation','2009 Excels/')

C:\Users\Administrator\Anaconda3\lib\site-packages\ipykernel_launcher.py:154: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy


done
done
done
Benchmark
